# Vietnamese Traffic Sign Dataset Audit (v2)

**Dataset:** the repository copy of the Kaggle `maitam/vietnamese-traffic-signs` dataset (Version 8).
**Purpose:** verify dataset integrity, label validity, the provided split (including near-duplicate leakage), class distribution and tiers, object sizes, and visual inspection.

**How to run:** open this notebook from the project and run all cells from top to bottom in a fresh kernel. By default, it locates `dataset/archive/` by searching the current working directory and its parents. If the archive is stored elsewhere, set `VNTS_ROOT` to the archive directory or to the repository/dataset directory containing it.

**Outputs** are written to `outputs/dataset_audit/` (CSV, PNG, JSON), which is ignored by Git. Set `VNTS_ARTEFACTS` to override the output directory. `audit_summary.json` holds headline numbers so the written report can be generated from recorded evidence.

**What changed from v1**
- Runs top to bottom with no hard-coded machine paths; output directories are created before anything is saved.
- Class counts are computed from the labels (v1 pasted them from an earlier output).
- One master table of boxes (`boxes.csv`) with image sizes and split; all later analysis uses it.
- Label validation (field count, ranges, duplicates), full image decoding, no bare `except`.
- Split checks now link the 25 empty labels to the 25 unlisted images, give per-class counts per split, and test for near-duplicates across train and test.
- Object sizes are also computed at training resolution (640 and 960 px), not only at original resolution.
- Visual inspection is seeded and class-aware: per-class contact sheets, full-image overlays, and a review log template.
- Expected values from the first audit are checked and printed as OK / MISMATCH.


## 1. Setup and configuration
All tunable settings are in this cell. Change them here, not further down.

In [ ]:
import json, os, platform, random, sys, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
from PIL import Image
from IPython.display import display


In [ ]:

# ---- reproducibility --------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = random.Random(SEED)

# ---- project and dataset locations ------------------------------------------
DATASET_HANDLE = "maitam/vietnamese-traffic-signs/versions/8"  # source provenance
DATASET_ROOT = os.environ.get("VNTS_ROOT")  # optional archive/repository override


def has_dataset_layout(path):
    return all((path / name).is_dir() for name in ("images", "labels", "split_dataset"))


def find_project_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if ((candidate / "dataset" / "archive").is_dir()
                and (candidate / "scripts" / "preprocessing" / "preprocessing.py").is_file()):
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Open the notebook from within the repository, "
        "or set VNTS_ROOT to the archive directory."
    )


PROJECT_ROOT = find_project_root()


def resolve_root():
    if DATASET_ROOT:
        supplied = Path(DATASET_ROOT).expanduser().resolve()
        candidates = (supplied, supplied / "archive", supplied / "dataset" / "archive")
        for candidate in candidates:
            if has_dataset_layout(candidate):
                return candidate.resolve()
        raise FileNotFoundError(
            f"VNTS_ROOT={supplied} does not contain images/, labels/, and split_dataset/."
        )

    root = PROJECT_ROOT / "dataset" / "archive"
    if not has_dataset_layout(root):
        raise FileNotFoundError(
            f"Expected the source dataset at {root}. Set VNTS_ROOT to an archive "
            "directory with images/, labels/, and split_dataset/ if it is stored elsewhere."
        )
    return root.resolve()


ROOT = resolve_root()

# ---- output folder (created immediately) -------------------------------------
_art = os.environ.get("VNTS_ARTEFACTS")
ARTEFACTS_DIR = Path(_art).expanduser() if _art else PROJECT_ROOT / "outputs" / "dataset_audit"
ARTEFACTS_DIR = ARTEFACTS_DIR.resolve()
ARTEFACTS_DIR.mkdir(parents=True, exist_ok=True)
SHEETS_DIR = ARTEFACTS_DIR / "class_sheets"
SHEETS_DIR.mkdir(exist_ok=True)

# ---- analysis settings -------------------------------------------------------
TIER_A_MIN, TIER_B_MIN = 150, 50      # tier thresholds in boxes (team decision, v1 audit)
NEAR_DUP_MAX_DIST = 6                 # dHash Hamming distance (of 64 bits) counted as near-duplicate
FULL_DECODE = True                    # fully decode every image to catch truncated files
TRAIN_IMGSZ = (640, 960)              # training resolutions used for the "size at training scale" check
N_PER_CLASS_SHEET = 12                # crops per class in the contact sheets
N_RANDOM_OVERLAYS = 25                # random full images drawn with their boxes
INSPECT_IDS = [1, 4, 37, 3, 18, 22, 28, 34]   # classes whose English names disagreed with the Vietnamese names

# Values seen in the first audit of Version 8. A MISMATCH is a warning, not an error.
EXPECTED = dict(images=3216, label_files=3216, boxes=8334, empty_labels=25, classes=52,
                train=2552, test=639, unlisted=25)

CHECKS = []
def check(name, observed, expected):
    status = "OK" if observed == expected else "MISMATCH"
    CHECKS.append(dict(name=name, observed=observed, expected=expected, status=status))
    print(f"{name:<24} observed={observed!s:<8} expected={expected!s:<8} {status}")

SUMMARY = dict(created=datetime.now().isoformat(timespec="seconds"), dataset_handle=DATASET_HANDLE,
               seed=SEED, root=str(ROOT),
               versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                             pillow=PIL.__version__, matplotlib=matplotlib.__version__))
print("project root  :", PROJECT_ROOT)
print("dataset root  :", ROOT)
print("artefacts dir :", ARTEFACTS_DIR)


## 2. Inventory and integrity
**Checks:** image and label files pair one-to-one; stems are unique; the three class-name files agree in length; every image decodes.
**Pass:** no unpaired files, no duplicate stems, no unreadable images, all three class files have the same number of lines.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png"}
img_files = [p for p in (ROOT / "images").rglob("*") if p.suffix.lower() in IMG_EXT]
lbl_files = list((ROOT / "labels").rglob("*.txt"))

dup_img = [s for s, n in Counter(p.stem for p in img_files).items() if n > 1]
dup_lbl = [s for s, n in Counter(p.stem for p in lbl_files).items() if n > 1]
imgs = {p.stem: p for p in img_files}
labels = {p.stem: p for p in lbl_files}

def read_lines(name):
    text = (ROOT / name).read_text(encoding="utf-8-sig", errors="replace")
    return [l.strip() for l in text.splitlines() if l.strip()]

classes, names_en, names_vi = read_lines("classes.txt"), read_lines("classes_en.txt"), read_lines("classes_vie.txt")
N_CLASSES = len(classes)
def safe(lst, i):
    return lst[i] if 0 <= i < len(lst) else ""

missing_labels = sorted(imgs.keys() - labels.keys())
missing_images = sorted(labels.keys() - imgs.keys())
print("image extensions:", dict(Counter(p.suffix.lower() for p in img_files)))
print("duplicate stems  : images", len(dup_img), "| labels", len(dup_lbl))
print("class files lines: classes.txt", len(classes), "| classes_en.txt", len(names_en), "| classes_vie.txt", len(names_vi))
check("images", len(img_files), EXPECTED["images"])
check("label_files", len(lbl_files), EXPECTED["label_files"])
check("classes", N_CLASSES, EXPECTED["classes"])
print("images without label:", len(missing_labels), "| labels without image:", len(missing_images))

In [ ]:
def image_info(stem, path):
    try:
        with Image.open(path) as im:
            w, h, mode = im.size[0], im.size[1], im.mode
            if FULL_DECODE:
                im.load()
        return dict(stem=stem, path=str(path.relative_to(ROOT)), img_w=w, img_h=h, mode=mode, ok=True, error="")
    except Exception as e:                      # record the failure instead of hiding it
        return dict(stem=stem, path=str(path.relative_to(ROOT)), img_w=0, img_h=0, mode="", ok=False, error=repr(e)[:200])

t0 = time.time()
img_df = pd.DataFrame([image_info(s, p) for s, p in sorted(imgs.items())])
print(f"{len(img_df)} images scanned in {time.time() - t0:.0f}s | unreadable: {(~img_df.ok).sum()}")
if (~img_df.ok).any():
    display(img_df[~img_df.ok])
print("\nmost common image sizes (width x height):")
display(img_df[img_df.ok].groupby(["img_w", "img_h"]).size().sort_values(ascending=False).head(10).rename("images").to_frame())

## 3. Labels: parse and validate
Every label line must be `class x_center y_center width height` (normalised). **Checks:** field count, numeric values, class id in range, coordinates in [0, 1], positive size, box inside the image, repeated lines.
Invalid lines are listed in `artefacts/label_issues.csv`, never silently skipped. An empty label file means "no annotated sign"; it is kept as an empty image (a candidate negative sample), not dropped.

In [ ]:
LABEL_TOL = 1e-3
rows, issues, empty_stems = [], [], set()

def add_issue(stem, line_no, kind, text):
    issues.append({"stem": stem, "line": line_no, "issue": kind, "text": text[:120]})

for stem, f in sorted(labels.items()):
    raw = f.read_text(encoding="utf-8-sig", errors="replace")
    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    if not lines:
        empty_stems.add(stem)
        continue
    seen = set()
    for no, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            add_issue(stem, no, "bad_field_count", line); continue
        try:
            cid_f = float(parts[0]); xc, yc, w, h = (float(v) for v in parts[1:])
        except ValueError:
            add_issue(stem, no, "non_numeric", line); continue
        if cid_f != int(cid_f):
            add_issue(stem, no, "non_integer_class", line); continue
        cid = int(cid_f)
        if not 0 <= cid < N_CLASSES:
            add_issue(stem, no, "class_out_of_range", line)
        if min(xc, yc, w, h) < 0 or max(xc, yc, w, h) > 1:
            add_issue(stem, no, "coord_out_of_range", line)
        if w <= 0 or h <= 0:
            add_issue(stem, no, "non_positive_size", line)
        if (xc - w / 2 < -LABEL_TOL or yc - h / 2 < -LABEL_TOL or
                xc + w / 2 > 1 + LABEL_TOL or yc + h / 2 > 1 + LABEL_TOL):
            add_issue(stem, no, "box_outside_image", line)
        if line in seen:
            add_issue(stem, no, "duplicate_line", line)
        seen.add(line)
        rows.append((stem, cid, xc, yc, w, h))

issues_df = pd.DataFrame(issues, columns=["stem", "line", "issue", "text"])
issues_df.to_csv(ARTEFACTS_DIR / "label_issues.csv", index=False)
boxes = pd.DataFrame(rows, columns=["stem", "class_id", "xc", "yc", "w", "h"])
boxes = boxes.merge(img_df[["stem", "img_w", "img_h"]], on="stem", how="left")
img_df["empty_label"] = img_df.stem.isin(empty_stems)
img_df["n_boxes"] = img_df.stem.map(boxes.groupby("stem").size()).fillna(0).astype(int)

check("boxes", len(boxes), EXPECTED["boxes"])
check("empty_labels", len(empty_stems), EXPECTED["empty_labels"])
print("label issues found:", len(issues_df))
if len(issues_df):
    display(issues_df.groupby("issue").size().rename("lines").to_frame())
    display(issues_df.head(15))
print("boxes per image (non-empty images):")
display(img_df[img_df.n_boxes > 0].n_boxes.describe().round(2).to_frame().T)

## 4. Provided train/test split
**Checks:** list sizes, shared entries, images in neither list, listed entries that do not exist, and whether the unlisted images are the empty-label images.
**Reading the result:** zero shared file names does **not** prove there is no leakage. Consecutive video frames have different names. Section 7 tests this directly.

In [ ]:
def read_list(name):
    text = (ROOT / "split_dataset" / name).read_text(encoding="utf-8-sig", errors="replace")
    return [Path(l.strip()).stem for l in text.splitlines() if l.strip()]

train_list, test_list = read_list("train_files.txt"), read_list("test_files.txt")
train, test = set(train_list), set(test_list)
overlap = train & test
unlisted = set(imgs) - train - test
listed_missing = (train | test) - set(imgs)

print("entries repeated inside a list: train", len(train_list) - len(train), "| test", len(test_list) - len(test))
check("train", len(train), EXPECTED["train"])
check("test", len(test), EXPECTED["test"])
check("unlisted", len(unlisted), EXPECTED["unlisted"])
print("shared by train and test:", len(overlap), "| listed but no image file:", len(listed_missing))
print(f"test share of listed images: {100 * len(test) / max(1, len(train | test)):.1f}%")

empty = set(empty_stems)
print("\nempty-label images vs unlisted images")
print("  empty AND unlisted      :", len(empty & unlisted))
print("  empty but listed        :", len(empty - unlisted))
print("  unlisted but not empty  :", len(unlisted - empty))

def split_of(s):
    if s in train and s in test: return "both"
    if s in train: return "train"
    if s in test: return "test"
    return "unlisted"

img_df["split"] = img_df.stem.map(split_of)
boxes["split"] = boxes.stem.map(split_of)
display(img_df.split.value_counts().rename("images").to_frame())

## 5. Class distribution and tiers
Tiers by **boxes** follow the team decision (A >= 150, B 50-149, C < 50). The same thresholds applied to **images** are shown for comparison, because splits and test sets are made of images.
The per-class table also shows boxes per split, so classes with no test boxes are visible immediately.

In [ ]:
def tier_of(n):
    return "A" if n >= TIER_A_MIN else ("B" if n >= TIER_B_MIN else "C")

valid = boxes[boxes.class_id.between(0, N_CLASSES - 1)]
cls_df = pd.DataFrame({"class_id": range(N_CLASSES)})
cls_df["classes_txt"] = [safe(classes, i) for i in cls_df.class_id]
cls_df["name_en"] = [safe(names_en, i) for i in cls_df.class_id]
cls_df["name_vi"] = [safe(names_vi, i) for i in cls_df.class_id]
cls_df["boxes"] = cls_df.class_id.map(valid.groupby("class_id").size()).fillna(0).astype(int)
cls_df["images"] = cls_df.class_id.map(valid.groupby("class_id").stem.nunique()).fillna(0).astype(int)
for sp in ("train", "test", "unlisted"):
    cls_df[f"{sp}_boxes"] = cls_df.class_id.map(valid[valid.split == sp].groupby("class_id").size()).fillna(0).astype(int)
cls_df["tier"] = cls_df.boxes.map(tier_of)
cls_df["tier_by_images"] = cls_df.images.map(tier_of)
cls_df.to_csv(ARTEFACTS_DIR / "class_tiers.csv", index=False)

tier_counts = cls_df.tier.value_counts().reindex(["A", "B", "C"]).fillna(0).astype(int)
print("Tier counts by boxes :", tier_counts.to_dict())
print("Tier counts by images:", cls_df.tier_by_images.value_counts().reindex(["A", "B", "C"]).fillna(0).astype(int).to_dict())
diff = cls_df[cls_df.tier != cls_df.tier_by_images]
print("classes whose tier differs (boxes vs images):")
display(diff[["class_id", "name_en", "boxes", "images", "tier", "tier_by_images"]])
print("classes with ZERO test boxes:", cls_df[cls_df.test_boxes == 0].class_id.tolist())
print("classes with ZERO train boxes:", cls_df[cls_df.train_boxes == 0].class_id.tolist())
display(cls_df.drop(columns=["name_vi"]))

order = cls_df.sort_values("boxes", ascending=False).reset_index(drop=True)
colors = order.tier.map({"A": "#2E7D32", "B": "#EF8F00", "C": "#9E9E9E"})
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(range(len(order)), order.boxes.clip(lower=0.5), color=colors)
ax.set_yscale("log")
ax.axhline(TIER_A_MIN, ls="--", c="k", lw=0.8); ax.axhline(TIER_B_MIN, ls="--", c="k", lw=0.8)
ax.set_xlabel("classes sorted by number of boxes"); ax.set_ylabel("boxes (log scale)")
ax.set_title(f"Boxes per class (green A >= {TIER_A_MIN}, orange B >= {TIER_B_MIN}, gray C)")
ax.set_xticks([])
fig.savefig(ARTEFACTS_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Object sizes
Sizes are computed from each box and **its own image's resolution**. Two views:
1. **Original resolution**, with COCO buckets (small < 32^2 px, medium < 96^2, large >= 96^2). Note that "small + medium" is everything except "large", so the share of *small* boxes is the informative number.
2. **Training resolution.** YOLO rescales the longer image side to `imgsz`. Equivalent box side = sqrt(area). Boxes below about 16 px at training scale are hard to detect (a rule of thumb, not a law), below about 8 px very hard.
If the image sizes in section 2 differ a lot, the second view matters more than the first.

In [ ]:
b = boxes[boxes.img_w.fillna(0) > 0].copy()
b["w_px"] = b.w * b.img_w
b["h_px"] = b.h * b.img_h
b["area_px"] = b.w_px * b.h_px
b["size"] = np.select([b.area_px < 32 ** 2, b.area_px < 96 ** 2], ["small", "medium"], "large")
b["eq_side"] = np.sqrt(b.area_px)
for sz in TRAIN_IMGSZ:
    b[f"eq_side_{sz}"] = b.eq_side * (sz / np.maximum(b.img_w, b.img_h))
b["tier"] = b.class_id.map(cls_df.set_index("class_id").tier)
S0 = TRAIN_IMGSZ[0]

counts = b["size"].value_counts().reindex(["small", "medium", "large"]).fillna(0).astype(int)
print("OBJECT SIZE (original resolution)")
display(pd.DataFrame({"count": counts, "percent": (100 * counts / len(b)).round(2)}))
print(f"width  median {b.w_px.median():.1f}px | min {b.w_px.min():.1f} | max {b.w_px.max():.1f}")
print(f"height median {b.h_px.median():.1f}px | min {b.h_px.min():.1f} | max {b.h_px.max():.1f}")

tiny = pd.DataFrame({f"at {sz}px": {"share < 16 px (%)": 100 * (b[f"eq_side_{sz}"] < 16).mean(),
                                    "share < 8 px (%)": 100 * (b[f"eq_side_{sz}"] < 8).mean(),
                                    "median equivalent side (px)": b[f"eq_side_{sz}"].median()} for sz in TRAIN_IMGSZ}).round(1)
print("\nOBJECT SIZE AT TRAINING RESOLUTION")
display(tiny)

print("size mix by split (share of boxes):")
display(b.groupby("split")["size"].value_counts(normalize=True).unstack().reindex(columns=["small", "medium", "large"]).round(3))
print("size mix by tier (share of boxes):")
display(b.groupby("tier")["size"].value_counts(normalize=True).unstack().reindex(columns=["small", "medium", "large"]).round(3))

g = b.groupby("class_id")
per_class = pd.DataFrame({
    "boxes": g.size(), "median_w_px": g.w_px.median().round(1), "median_h_px": g.h_px.median().round(1),
    "small_share": g["size"].apply(lambda s: (s == "small").mean()).round(2),
    f"share_below_16px_at_{S0}": g[f"eq_side_{S0}"].apply(lambda s: (s < 16).mean()).round(2),
})
per_class["name_en"] = per_class.index.map(lambda i: safe(names_en, i))
per_class["tier"] = per_class.index.map(cls_df.set_index("class_id").tier)
per_class.to_csv(ARTEFACTS_DIR / "class_size_stats.csv")
print(f"classes with the highest share of boxes below 16 px at {S0}px (Tier A and B only):")
display(per_class[per_class.tier.isin(["A", "B"])].sort_values(f"share_below_16px_at_{S0}", ascending=False).head(10))

b[["stem", "split", "class_id", "tier", "xc", "yc", "w", "h", "img_w", "img_h", "w_px", "h_px", "area_px", "size"]
  + [f"eq_side_{sz}" for sz in TRAIN_IMGSZ]].to_csv(ARTEFACTS_DIR / "boxes.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(b.w_px, bins=50); axes[0].set_title("box width (px, original)")
axes[1].hist(b.h_px, bins=50); axes[1].set_title("box height (px, original)")
axes[2].hist(b[f"eq_side_{S0}"], bins=50); axes[2].axvline(16, c="r", ls="--")
axes[2].set_title(f"equivalent side at {S0}px (red = 16 px)")
fig.savefig(ARTEFACTS_DIR / "box_size_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Near-duplicate leakage check
Identical file names are not the only way a split can leak: consecutive frames from one drive look almost the same. This section hashes every image (**dHash**, 64 bits, no extra dependency) and measures how close each test image is to the nearest train image.

**How to read it**
- The table compares *test -> nearest train* with a **reference**: *train -> nearest other train*. If the two rows look alike, the split is random over similar frames, so test images have near-twins in train. If test is far from train but train images have close neighbours inside train, the split respects scenes.
- `NEAR_DUP_MAX_DIST` is a heuristic. Look at the side-by-side pairs before trusting any count: similar road scenes can hash close without being the same frame.
- `groups.csv` assigns each image to a group of near-duplicates. Use these groups when creating the validation split, so no group is split across train and validation.

In [ ]:
RESAMPLE = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS

def dhash_bits(path, size=8):
    with Image.open(path) as im:
        im.draft("L", (size * 16, size * 16))          # fast reduced-size JPEG decode
        g = im.convert("L").resize((size + 1, size), RESAMPLE)
    a = np.asarray(g, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).astype(np.uint8).ravel()

ok_df = img_df[img_df.ok].reset_index(drop=True)
t0 = time.time()
H = np.stack([dhash_bits(imgs[s]) for s in ok_df.stem])
print(f"hashed {len(H)} images in {time.time() - t0:.0f}s")
Hf = H.astype(np.float32); Hn = 1.0 - Hf
n = len(ok_df)

def hamming(rows_idx, cols_idx):
    return Hf[rows_idx] @ Hn[cols_idx].T + Hn[rows_idx] @ Hf[cols_idx].T

# all pairs within the distance threshold
pairs, CH = [], 800
for i0 in range(0, n, CH):
    idx = np.arange(i0, min(n, i0 + CH))
    D = hamming(idx, np.arange(n))
    ii, jj = np.where(D <= NEAR_DUP_MAX_DIST)
    for a_loc, b_idx in zip(ii, jj):
        a_idx = a_loc + i0
        if a_idx < b_idx:
            pairs.append((a_idx, b_idx, int(D[a_loc, b_idx])))
print(f"image pairs within Hamming distance {NEAR_DUP_MAX_DIST}: {len(pairs)}")

# union-find groups
parent = list(range(n))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a_idx, b_idx, d in pairs:
    ra, rb = find(a_idx), find(b_idx)
    if ra != rb:
        parent[rb] = ra
ok_df["group_id"] = pd.factorize(pd.Series([find(i) for i in range(n)]))[0]
ok_df["group_size"] = ok_df.groupby("group_id").stem.transform("size")
span = ok_df.groupby("group_id").split.agg(lambda s: frozenset(s) - {"unlisted"})
n_span = int(sum(1 for s in span if {"train", "test"} <= s))
print(f"groups of 2+ images: {(ok_df.groupby('group_id').size() > 1).sum()} | groups that contain both train and test images: {n_span}")

pairs_df = pd.DataFrame([(ok_df.stem[a], ok_df.split[a], ok_df.stem[b], ok_df.split[b], d) for a, b, d in pairs],
                        columns=["stem_a", "split_a", "stem_b", "split_b", "distance"])
pairs_df.to_csv(ARTEFACTS_DIR / "near_duplicate_pairs.csv", index=False)
ok_df[["stem", "split", "group_id", "group_size"]].to_csv(ARTEFACTS_DIR / "groups.csv", index=False)

# test -> nearest train, versus train -> nearest other train (reference)
split_arr = ok_df.split.values
T, R = np.where(split_arr == "test")[0], np.where(split_arr == "train")[0]

def min_dist(a_idx, b_idx, exclude_self=False):
    out = []
    for k0 in range(0, len(a_idx), CH):
        sub = a_idx[k0:k0 + CH]
        D = hamming(sub, b_idx)
        if exclude_self:
            D = np.where(sub[:, None] == b_idx[None, :], 99, D)
        out.append(D.min(axis=1))
    return np.concatenate(out) if out else np.array([])

leak = {}
if len(T) and len(R):
    d_tt = min_dist(T, R)
    ref = np.random.RandomState(SEED).choice(R, size=min(len(T), len(R)), replace=False)
    d_rr = min_dist(ref, R, exclude_self=True)
    deltas = [0, 2, 4, 6, 8, 10, 12]
    table = pd.DataFrame({"test -> nearest train (%)": [100 * (d_tt <= d).mean() for d in deltas],
                          "train -> nearest other train (%)": [100 * (d_rr <= d).mean() for d in deltas]},
                         index=[f"<= {d}" for d in deltas]).round(1)
    table.index.name = "Hamming distance"
    print("share of images that have a neighbour within the distance:")
    display(table)
    leak = dict(threshold=NEAR_DUP_MAX_DIST,
                test_with_train_neighbour_pct=round(100 * float((d_tt <= NEAR_DUP_MAX_DIST).mean()), 1),
                train_with_other_train_neighbour_pct=round(100 * float((d_rr <= NEAR_DUP_MAX_DIST).mean()), 1),
                groups_spanning_train_and_test=n_span)
    table.to_csv(ARTEFACTS_DIR / "leakage_sensitivity.csv")

    # the closest test/train pairs, side by side, for manual judgement
    K = min(8, len(T))
    closest = np.argsort(d_tt)[:K]
    fig, axes = plt.subplots(4, 4, figsize=(14, 9))
    for ax in axes.ravel():
        ax.axis("off")
    for k, ti in enumerate(closest):
        t_idx = T[ti]
        j = R[int(np.argmin(hamming(np.array([t_idx]), R)[0]))]
        for slot, (idx, tag) in enumerate(((t_idx, "TEST"), (j, "TRAIN"))):
            ax = axes.ravel()[2 * k + slot]
            with Image.open(imgs[ok_df.stem[idx]]) as im:
                im = im.convert("RGB"); im.thumbnail((420, 420))
            ax.imshow(im); ax.set_title(f"{tag} | dist {int(d_tt[ti])}\n{ok_df.stem[idx][:26]}", fontsize=7)
    fig.suptitle("Closest test images and their nearest train image", fontsize=12)
    fig.savefig(ARTEFACTS_DIR / "leakage_closest_pairs.png", dpi=130, bbox_inches="tight")
    plt.show()
SUMMARY["leakage"] = leak

In [ ]:
# ===== LEAKAGE REVIEW: is the ~55% real? (run right after the leakage cell) =====
import re

# 1) Flag "low-information" images. A flat, dark or blurry image gives an almost all-0 (or all-1) dHash,
#    so unrelated featureless images look identical to the hash. Natural images have about 32 ones out of 64.
ones = H.sum(axis=1)
ok_df["low_info"] = (ones <= 10) | (ones >= 54)
ok_df["n_boxes"] = ok_df.stem.map(img_df.set_index("stem").n_boxes).fillna(0).astype(int)
print("low-information images:", int(ok_df.low_info.sum()), "of", n)
display(ok_df.groupby("split").agg(images=("stem", "size"), low_info=("low_info", "sum"),
                                   no_box=("n_boxes", lambda s: int((s == 0).sum()))))

# 2) Neighbour rates again, split by image type (T, R, d_tt come from the leakage cell)
d_rr_all = min_dist(R, R, exclude_self=True)       # every train image -> nearest OTHER train image
low_T, low_R = ok_df.low_info.values[T], ok_df.low_info.values[R]

def pct(d, keep, delta):
    d = d[keep]
    return round(100 * float((d <= delta).mean()), 1) if len(d) else float("nan")

rate_tbl = pd.DataFrame([{"distance <=": dl,
                          "test->train (all)": pct(d_tt, np.ones(len(T), bool), dl),
                          "test->train (not low-info)": pct(d_tt, ~low_T, dl),
                          "train->train (all)": pct(d_rr_all, np.ones(len(R), bool), dl),
                          "train->train (not low-info)": pct(d_rr_all, ~low_R, dl)} for dl in (0, 2, 4, 6)]
                        ).set_index("distance <=")
display(rate_tbl)

# 3) Nearest train image for EVERY test image, and what the file names say
near = []
for k0 in range(0, len(T), CH):
    sub = T[k0:k0 + CH]
    Dm = hamming(sub, R)
    near.append(np.column_stack([R[Dm.argmin(axis=1)], Dm.min(axis=1)]))
near = np.vstack(near)
cross = pd.DataFrame({"test_idx": T, "train_idx": near[:, 0].astype(int), "distance": near[:, 1].astype(int)})
cross["test_stem"] = ok_df.stem.values[cross.test_idx]
cross["train_stem"] = ok_df.stem.values[cross.train_idx]
cross["low_info"] = ok_df.low_info.values[cross.test_idx] | ok_df.low_info.values[cross.train_idx]

def split_name(s):
    m = re.match(r"^(.*?)(\d+)$", s)
    return (m.group(1), int(m.group(2))) if m else (s, None)

parts_t, parts_r = cross.test_stem.map(split_name), cross.train_stem.map(split_name)
cross["num_gap"] = [abs(x[1] - y[1]) if (x[0] == y[0] and x[1] is not None and y[1] is not None) else np.nan
                    for x, y in zip(parts_t, parts_r)]
close = cross[cross.distance <= NEAR_DUP_MAX_DIST]
print("example linked pairs (closest first):")
display(cross.sort_values("distance").head(8)[["test_stem", "train_stem", "distance"]])
print(f"linked pairs (distance <= {NEAR_DUP_MAX_DIST}): {len(close)} | with consecutive-looking names "
      f"(same prefix, number gap <= 5): {int((close.num_gap <= 5).sum())} | involving a low-info image: {int(close.low_info.sum())}")

# 4) Seeded, stratified sample of cross-split pairs for manual judgement
BANDS, N_PER_BAND = [(0, 2), (3, 4), (5, 6)], 20
parts = []
for lo, hi in BANDS:
    pool = cross[(cross.distance >= lo) & (cross.distance <= hi)]
    take = pool.sample(min(N_PER_BAND, len(pool)), random_state=SEED) if len(pool) else pool
    parts.append(take.assign(band=f"{lo}-{hi}"))
review = pd.concat(parts, ignore_index=True)
review["pair_id"] = np.arange(len(review))

def show_pairs(df, title, path, per_row=3):
    rows_n = max(1, int(np.ceil(len(df) / per_row)))
    fig, axes = plt.subplots(rows_n, per_row * 2, figsize=(15, 2.4 * rows_n + 0.6), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for k, (_, r) in enumerate(df.iterrows()):
        row, col = divmod(k, per_row)
        for slot, (idx, tag) in enumerate(((r.test_idx, "TEST"), (r.train_idx, "TRAIN"))):
            with Image.open(imgs[ok_df.stem[int(idx)]]) as im:
                im = im.convert("RGB"); im.thumbnail((320, 320))
            ax = axes[row, 2 * col + slot]
            ax.imshow(im)
            ax.set_title(f"#{int(r.pair_id)} TEST  d={int(r.distance)}" if slot == 0 else "TRAIN", fontsize=7)
    fig.suptitle(title, fontsize=11)
    fig.savefig(path, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)

for lo, hi in BANDS:
    sel = review[review.band == f"{lo}-{hi}"]
    if len(sel):
        show_pairs(sel, f"Distance {lo}-{hi}: same scene, similar-looking, or blank?",
                   ARTEFACTS_DIR / f"leakage_review_{lo}_{hi}.png")

# 5) Review log (never overwritten). verdict: same | similar | blank | unsure
#    same    = same place / near-identical photo (e.g. consecutive video frames)
#    similar = different place that merely looks alike
#    blank   = featureless image (dark, blurry, empty road)
log_path = ARTEFACTS_DIR / "leakage_review_log.csv"
if not log_path.exists():
    out = review[["pair_id", "band", "test_stem", "train_stem", "distance", "low_info"]].copy()
    out["verdict"] = ""
    out["notes"] = ""
    out.to_csv(log_path, index=False)
    print("created", log_path)
else:
    print("kept existing", log_path)

def summarise_review(path=log_path):
    r = pd.read_csv(path)
    r = r[r.verdict.notna() & (r.verdict.astype(str).str.strip() != "")]
    if r.empty:
        print("no verdicts filled in yet"); return
    print("counts:");  display(pd.crosstab(r.band, r.verdict))
    print("shares:");  display(pd.crosstab(r.band, r.verdict, normalize="index").round(2))

## 8. Visual inspection
v1 inspected 60 random crops, which cannot support "no significant annotation problems": the sample was unseeded, ignored class, and crops cannot show *missing* boxes. This section is designed so the review can be divided among the team and evidence is kept:
1. **Size galleries** (seeded; kept from v1).
2. **Per-class contact sheets**: up to `N_PER_CLASS_SHEET` crops of every class, titled with the English *and* Vietnamese names. Classes in `INSPECT_IDS` are shown inline; open the others in `artefacts/class_sheets/`.
3. **Full-image overlays**: the empty-label images (to confirm there is really no sign) and random images with their boxes drawn (to spot unlabelled signs).
4. **Review log** (`visual_review_log.csv`): record what was checked and what was wrong. It is created once and never overwritten.

In [ ]:
def crop_box(stem, xc, yc, bw, bh, margin=0.15):
    with Image.open(imgs[stem]) as im:
        im = im.convert("RGB")
    W, H = im.size
    cx, cy, w, h = xc * W, yc * H, bw * W, bh * H
    mx, my = w * margin, h * margin
    x1, y1 = max(0, int(cx - w / 2 - mx)), max(0, int(cy - h / 2 - my))
    x2, y2 = min(W, int(cx + w / 2 + mx)), min(H, int(cy + h / 2 + my))
    return im.crop((x1, y1, x2, y2)) if x2 > x1 and y2 > y1 else None

def draw_tiles(df, title, path, cols=5, label_fn=None, show=True):
    rows_n = max(1, int(np.ceil(len(df) / cols)))
    fig, axes = plt.subplots(rows_n, cols, figsize=(2.6 * cols, 2.7 * rows_n + 0.8), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), df.iterrows()):
        crop = crop_box(r.stem, r.xc, r.yc, r.w, r.h)
        if crop is not None:
            ax.imshow(crop)
            ax.set_title(label_fn(r) if label_fn else str(r.class_id), fontsize=7)
    fig.suptitle(title, fontsize=11)
    fig.savefig(path, dpi=130, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)

# 8.1 size galleries (seeded)
for bucket in ("small", "medium", "large"):
    pool = b[b["size"] == bucket]
    sample = pool.sample(min(20, len(pool)), random_state=SEED)
    draw_tiles(sample, f"{bucket.capitalize()} objects (seed {SEED})", ARTEFACTS_DIR / f"gallery_{bucket}.png",
               label_fn=lambda r: f"{r.class_id} {safe(names_en, r.class_id)[:18]}\n{int(r.w_px)}x{int(r.h_px)} px")

In [ ]:
# 8.2 per-class contact sheets (all classes saved; INSPECT_IDS shown inline)
for cid in range(N_CLASSES):
    sub = boxes[boxes.class_id == cid].drop_duplicates("stem")
    if sub.empty:
        continue
    sub = sub.sample(min(N_PER_CLASS_SHEET, len(sub)), random_state=SEED)
    n_boxes = int(cls_df.loc[cid, "boxes"])
    draw_tiles(sub, f"class {cid} | EN: {safe(names_en, cid)} | VI: {safe(names_vi, cid)} | {n_boxes} boxes",
               SHEETS_DIR / f"class_{cid:02d}.png", cols=6,
               label_fn=lambda r: f"{r.stem[:14]}", show=cid in INSPECT_IDS)
print(f"saved {len(list(SHEETS_DIR.glob('class_*.png')))} class sheets to {SHEETS_DIR}")

In [ ]:
# 8.3 full-image overlays
def draw_overlays(stems, title, path, cols=5):
    rows_n = max(1, int(np.ceil(len(stems) / cols)))
    fig, axes = plt.subplots(rows_n, cols, figsize=(3.2 * cols, 2.4 * rows_n + 0.6), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, stem in zip(axes.ravel(), stems):
        with Image.open(imgs[stem]) as im:
            im = im.convert("RGB"); W, H = im.size; im.thumbnail((520, 520))
        ax.imshow(im, extent=(0, W, H, 0))
        for _, r in boxes[boxes.stem == stem].iterrows():
            ax.add_patch(plt.Rectangle(((r.xc - r.w / 2) * W, (r.yc - r.h / 2) * H), r.w * W, r.h * H,
                                       fill=False, ec="lime", lw=1.2))
            ax.text((r.xc - r.w / 2) * W, (r.yc - r.h / 2) * H, str(int(r.class_id)), color="yellow", fontsize=7,
                    va="bottom", bbox=dict(fc="black", alpha=0.5, pad=0.5, ec="none"))
        ax.set_title(stem[:24], fontsize=7)
    fig.suptitle(title, fontsize=11)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)

empty_sorted = sorted(empty_stems)[:25]
if empty_sorted:
    draw_overlays(empty_sorted, f"Empty-label images ({len(empty_stems)} total, first {len(empty_sorted)} shown): is any sign visible?",
                  ARTEFACTS_DIR / "overlay_empty_images.png")
with_boxes = sorted(img_df[img_df.n_boxes > 0].stem)
rand_stems = RNG.sample(with_boxes, min(N_RANDOM_OVERLAYS, len(with_boxes)))
draw_overlays(rand_stems, f"Random images with boxes (seed {SEED}): any sign without a box?",
              ARTEFACTS_DIR / "overlay_random_images.png")

# 8.4 review log template (never overwritten)
log_path = ARTEFACTS_DIR / "visual_review_log.csv"
if not log_path.exists():
    log = cls_df[["class_id", "name_en", "name_vi", "boxes"]].copy()
    log["sheet"] = log.class_id.map(lambda i: f"class_sheets/class_{i:02d}.png")
    for col in ("reviewed_by", "crops_checked", "wrong_label", "missing_box_seen", "notes"):
        log[col] = ""
    log.to_csv(log_path, index=False)
    print("created", log_path)
else:
    print("kept existing", log_path)

## 9. Summary and artefacts

Writes `audit_summary.json` to `outputs/dataset_audit/` (or the directory set by `VNTS_ARTEFACTS`). Copy reported numbers from this file and keep interpretation separate from evidence. Conclusions about annotation quality are only as strong as the review log behind them.


In [ ]:
SUMMARY.update(
    integrity=dict(images=int(len(img_df)), label_files=int(len(lbl_files)), unreadable_images=int((~img_df.ok).sum()),
                   images_without_label=len(missing_labels), labels_without_image=len(missing_images),
                   empty_label_files=len(empty_stems), label_issue_lines=int(len(issues_df)),
                   boxes=int(len(boxes)), classes=int(N_CLASSES)),
    split=dict(train=len(train), test=len(test), shared=len(overlap), unlisted=len(unlisted),
               empty_and_unlisted=len(empty & unlisted), listed_but_missing=len(listed_missing),
               classes_with_zero_test_boxes=cls_df[cls_df.test_boxes == 0].class_id.tolist()),
    tiers=dict(thresholds=dict(A=TIER_A_MIN, B=TIER_B_MIN), by_boxes=tier_counts.to_dict(),
               by_images=cls_df.tier_by_images.value_counts().reindex(["A", "B", "C"]).fillna(0).astype(int).to_dict(),
               differ=diff.class_id.tolist()),
    sizes=dict(original={k: int(v) for k, v in counts.items()},
               small_pct=round(100 * counts["small"] / len(b), 2),
               median_w_px=round(float(b.w_px.median()), 1), median_h_px=round(float(b.h_px.median()), 1),
               share_below_16px={f"{sz}px": round(100 * float((b[f"eq_side_{sz}"] < 16).mean()), 1) for sz in TRAIN_IMGSZ}),
    checks=CHECKS)
with open(ARTEFACTS_DIR / "audit_summary.json", "w", encoding="utf-8") as f:
    json.dump(SUMMARY, f, indent=2, default=str)

print("expected-value checks:", sum(c["status"] == "OK" for c in CHECKS), "OK /",
      sum(c["status"] == "MISMATCH" for c in CHECKS), "MISMATCH")
print("\nartefacts written:")
for p in sorted(ARTEFACTS_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name:<32} {p.stat().st_size / 1024:8.1f} KB")
print(f"  class_sheets/ ({len(list(SHEETS_DIR.glob('*.png')))} files)")

In [ ]:
# ===== FINAL SPLIT (run AFTER the manual review; set DELTA from it) =====
import warnings
from sklearn.model_selection import StratifiedGroupKFold

DELTA = 4                                   # <- choose from the review (must be <= NEAR_DUP_MAX_DIST)
assert DELTA <= NEAR_DUP_MAX_DIST, "recompute `pairs` with a larger NEAR_DUP_MAX_DIST first"
IGNORE_LOW_INFO_LINKS = True                # featureless images are not treated as duplicates of each other
low = ok_df.low_info.values

# (a) clean test = provided test images whose nearest train image is farther than DELTA
linked = (cross.distance <= DELTA) & ~(IGNORE_LOW_INFO_LINKS & cross.low_info)
clean_test = set(cross.test_stem[~linked])
print(f"clean test: {len(clean_test)} of {len(T)} provided test images ({100 * len(clean_test) / max(1, len(T)):.1f}%)")

# (b) near-duplicate groups inside train only, at DELTA
parent2 = {int(i): int(i) for i in R}
def find2(x):
    while parent2[x] != x:
        parent2[x] = parent2[parent2[x]]
        x = parent2[x]
    return x
for a_i, b_i, d in pairs:
    if d <= DELTA and split_arr[a_i] == "train" and split_arr[b_i] == "train" \
            and not (IGNORE_LOW_INFO_LINKS and (low[a_i] or low[b_i])):
        ra, rb = find2(int(a_i)), find2(int(b_i))
        if ra != rb:
            parent2[rb] = ra
tr = ok_df.iloc[R][["stem"]].copy()
tr["group"] = [find2(int(i)) for i in R]
sizes = tr.groupby("group").size().sort_values(ascending=False)
print("largest train groups:", sizes.head(5).tolist(),
      "| share of train in the largest group:", f"{100 * sizes.iloc[0] / len(tr):.1f}%")

# (c) validation = about 1/6 of train, cut by GROUPS (a group never spans train and val),
#     roughly balanced on each image's rarest class
rank = cls_df.set_index("class_id").boxes
rare = valid.groupby("stem").class_id.apply(lambda s: int(min(s, key=lambda c: rank[c])))
tr["y"] = tr.stem.map(rare).fillna(-1).astype(int)
sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _, val_pos = next(sgkf.split(tr, tr.y, groups=tr.group))
val_set = set(tr.stem.iloc[val_pos])

# (d) write the manifest
final = ok_df[["stem", "split"]].rename(columns={"split": "provided"}).copy()
final["role"] = ["val" if (p == "train" and s in val_set) else p if p in ("train", "test") else "unlisted"
                 for s, p in zip(final.stem, final.provided)]
final["clean_test"] = final.stem.isin(clean_test)
final.to_csv(ARTEFACTS_DIR / "split_v1.csv", index=False)
display(final.role.value_counts().rename("images").to_frame())

# (e) boxes per class in every role; flag thin classes
bx = valid.merge(final[["stem", "role", "clean_test"]], on="stem")
tab = pd.DataFrame({"train": bx[bx.role == "train"].groupby("class_id").size(),
                    "val": bx[bx.role == "val"].groupby("class_id").size(),
                    "test": bx[bx.role == "test"].groupby("class_id").size(),
                    "clean_test": bx[(bx.role == "test") & bx.clean_test].groupby("class_id").size()}
                   ).reindex(range(N_CLASSES)).fillna(0).astype(int)
tab["tier"] = cls_df.set_index("class_id").tier
tab["name_en"] = [safe(names_en, i) for i in tab.index]
tab.to_csv(ARTEFACTS_DIR / "split_v1_class_counts.csv")
ab = tab[tab.tier.isin(["A", "B"])]
display(ab)
print("Tier A/B classes with < 10 boxes in val        :", ab.index[ab.val < 10].tolist())
print("Tier A/B classes with < 10 boxes in clean test :", ab.index[ab.clean_test < 10].tolist())

In [ ]:
summarise_review(log_path)